In [2]:
import numpy as np
import pandas as pd
import networkx as nx
import pickle
import re
from pathlib import Path
import io


In [3]:

# %%
# This function is copied directly from your netsim.txt file.
# It is used to create the time-expanded ground truth DAG.

def unroll_dag_for_time_window(static_net: np.ndarray, k: int) -> nx.DiGraph:
    """
    Unrolls a static causal graph into a time-expanded DiGraph over a window k.

    This function builds a graph based on two key hypotheses:
    1.  Self-Markov Hypothesis: Every node i at time t-1 causes itself at time t.
        (Edge: i_{t-1} -> i_{t})
    2.  Lag 1 Causal Hypothesis: A static link i -> j implies a causal effect
        from node i at time t-1 to node j at time t.
        (Edge: i_{t-1} -> j_{t})

    Args:
        static_net: An (N x N) numpy array representing the static graph,
                    where static_net[i, j] > 0 indicates a causal link from i to j.
        k: The number of past time steps to unroll (e.g., k=3 unrolls back to t-3).

    Returns:
        A NetworkX DiGraph object representing the full, unrolled causal structure.
        Nodes are named in the format 'nodeID_t-lag'.
    """
    if k < 1:
        raise ValueError("Time window k must be at least 1.")

    N_nodes = static_net.shape[0]
    G = nx.DiGraph()

    # The main loop iterates through each time step transition, from the past to the present.
    # We are creating edges that connect slice (t-lag) to slice (t-lag+1).
    for lag in range(k, 0, -1):  # Iterates from k down to 1
        source_time_slice = lag      # e.g., t-3
        target_time_slice = lag - 1  # e.g., t-2

        # Loop through all nodes to add edges for this time transition
        for i in range(N_nodes):
            source_node_i = f"{i}_t-{source_time_slice}"

            # Hypothesis 2: Self-Markov (auto-causal) links
            # Every node i at t-lag causes itself at t-(lag-1)
            target_node_i = f"{i}_t-{target_time_slice}"
            G.add_edge(source_node_i, target_node_i, type='auto')

            # Hypothesis 1: Lag 1 Causal (cross-causal) links
            # Check for causal effects originating from node i
            for j in range(N_nodes):
                # If the static graph has a link i -> j...
                if i != j and static_net[i, j] > 0:
                    # ...create an edge from i at t-lag to j at t-(lag-1)
                    target_node_j = f"{j}_t-{target_time_slice}"
                    weight = static_net[i, j]
                    G.add_edge(source_node_i, target_node_j, weight=weight, type='cross')
    
    # Ensure nodes in the final time slice (t-0) are created, even if they have no incoming edges.
    # Note: add_edge already does this, but this is an explicit safeguard.
    for i in range(N_nodes):
        G.add_node(f"{i}_t-0")
        
    return G


# %%
# New functions to handle DREAM3 data formats

def load_dream3_gold_standard(filepath: Path) -> np.ndarray:
    """
    Parses a DREAM3 gold standard file to create a static adjacency matrix.

    Args:
        filepath: Path to the gold standard .txt file.

    Returns:
        A numpy array (N x N) representing the static adjacency matrix.
    """
    edges = []
    max_gene_num = 0
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 3 and parts[2] == '1':
                # Extract gene numbers (e.g., from 'G10' to 9)
                source_gene = int(re.search(r'G(\d+)', parts[0]).group(1))
                target_gene = int(re.search(r'G(\d+)', parts[1]).group(1))
                
                # Convert to 0-based index
                source_idx = source_gene - 1
                target_idx = target_gene - 1
                
                edges.append((source_idx, target_idx))
                max_gene_num = max(max_gene_num, source_gene, target_gene)

    num_nodes = max_gene_num
    adj_matrix = np.zeros((num_nodes, num_nodes), dtype=int)
    for src, tgt in edges:
        adj_matrix[src, tgt] = 1
        
    print(f"Loaded gold standard from {filepath}. Found {len(edges)} edges among {num_nodes} nodes.")
    return adj_matrix


def load_dream3_trajectories(filepath: Path):
    """
    Parses a DREAM3 trajectories file, which contains multiple concatenated
    time-series experiments.

    Args:
        filepath: Path to the trajectories .tsv file.

    Returns:
        A list of numpy arrays, where each array is a separate time-series
        observation (timepoints x nodes).
    """
    with open(filepath, 'r') as f:
        content = f.read()
    
    # Each experiment is separated by the header line '"Time"\t"G1"\t...'
    # We split the file content by this header to isolate experiments.
    # The first element after split will be empty, so we skip it.
    exp_blocks = content.split('"Time"\t"G1"')[1:]
    
    all_trajectories = []
    for block in exp_blocks:
        # Re-add a simplified header for pandas to parse correctly
        header = "Time\tG1" + block.split('\n')[0]
        data_rows = '\n'.join(block.split('\n')[1:])
        full_block_text = header + '\n' + data_rows
        
        # Use pandas to easily read the tab-separated data
        df = pd.read_csv(io.StringIO(full_block_text), sep='\t')
        
        # Drop the time column and convert to numpy array
        # Ensure all data is numeric, coercing errors to NaN and then filling with 0
        df = df.drop(columns=['Time'])
        trajectory_data = df.apply(pd.to_numeric, errors='coerce').fillna(0).values
        
        # Only add non-empty trajectories
        if trajectory_data.shape[0] > 0:
            all_trajectories.append(trajectory_data)

    print(f"Loaded trajectories from {filepath}. Found {len(all_trajectories)} separate experiments.")
    return all_trajectories

In this case, since we want to avoid 50 or 100 dimensions for the sake of simplicity, we have to stick to 10. If we stick to 10, we only have 5 actual observations. This implies

In [7]:
import os 
root_ground_truth = './raw/gold standard/'
root_observations = './raw/training data/InSilicoSize50/'
generative_process_counter = 51

observations = {generative_process_counter: {}}
dags = {generative_process_counter: {}}

observation_counter = 0
for file in os.listdir(root_ground_truth):
    if '50_' in file:
        
        size_namefile = file.split('_')[1] # InSilicoSize10
        bacteria_namefile = file.split('_')[2].split('.')[0] # Ecoli1

        gold_standard_file = root_ground_truth + file
        trajectories_file = root_observations + f"{size_namefile}-{bacteria_namefile}-trajectories.tsv"


        # --- Data Loading ---
        static_adj_matrix = load_dream3_gold_standard(gold_standard_file)
        observation = load_dream3_trajectories(trajectories_file)
        print('Len of observation:', len(observation))
        # The ground truth DAG is the same for all observations from this process
        # We use k=3 to match the unrolling window in netsim.txt
        unrolled_G = unroll_dag_for_time_window(static_adj_matrix, k=3)
        num_nodes, num_edges = len(unrolled_G.nodes), len(unrolled_G.edges)
        print(f"Generated time-unrolled DAG with {num_nodes} nodes and {num_edges} edges (k=3).")

        observations[generative_process_counter][observation_counter] = observation[0]
        dags[generative_process_counter][observation_counter] = unrolled_G
        
        observation_counter += 1



Loaded gold standard from ./raw/gold standard/DREAM3GoldStandard_InSilicoSize50_Ecoli2.txt. Found 82 edges among 50 nodes.
Loaded trajectories from ./raw/training data/InSilicoSize50/InSilicoSize50-Ecoli2-trajectories.tsv. Found 1 separate experiments.
Len of observation: 1
Generated time-unrolled DAG with 200 nodes and 396 edges (k=3).
Loaded gold standard from ./raw/gold standard/DREAM3GoldStandard_InSilicoSize50_Ecoli1.txt. Found 62 edges among 50 nodes.
Loaded trajectories from ./raw/training data/InSilicoSize50/InSilicoSize50-Ecoli1-trajectories.tsv. Found 1 separate experiments.
Len of observation: 1
Generated time-unrolled DAG with 200 nodes and 336 edges (k=3).
Loaded gold standard from ./raw/gold standard/DREAM3GoldStandard_InSilicoSize50_Yeast3.txt. Found 173 edges among 50 nodes.
Loaded trajectories from ./raw/training data/InSilicoSize50/InSilicoSize50-Yeast3-trajectories.tsv. Found 1 separate experiments.
Len of observation: 1
Generated time-unrolled DAG with 200 nodes and

Yep, this DAG is coherent with the corresponding file content. 

In [8]:
# --- Save the final output ---
output_filename = 'dream3_50.pkl'
final_data_structure = (observations, dags, None)

with open(output_filename, 'wb') as f:
    pickle.dump(final_data_structure, f)
